In [ ]:
# Clone the repository
!git clone https://github.com/nsuurmey/block-party.git

# Install dependencies
!pip install -r block-party/mvp0/requirements.txt

# Add the repo to the Python path so boem_loader.py can be imported
import sys
sys.path.insert(0, 'block-party/mvp0')
sys.path.insert(0, 'block-party')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Symlink your Drive data folder into the repo's data directory
import os
os.symlink(
    '/content/drive/MyDrive/block-party-data',  # adjust to your Drive path
    'block-party/data'
)

# 00 — Data Loading & Sanity Checks

Load BOEM lease-sale files and the OCS block shapefile using `boem_loader`.

**Sales loaded:** 198 (available now), 257, 261, Dec 2025 (add data directories once downloaded).

**Outputs:** row counts, column heads, basic spatial plot.

In [ ]:
import sys, os

# Add mvp0/ to path so we can import the helper module
import os, sys

ROOT = '/content/block-party'
sys.path.insert(0, os.path.join(ROOT, 'mvp0'))


import boem_loader as bl
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 1. Define sale directories

Each BOEM sale ZIP should be unzipped into its own folder under `data/lease-sales/`.

| Directory | Sale | Status |
|-----------|------|--------|
| `sale_198` | Sale 198 | available |
| `sale_257` | Sale 257 (Aug 2023) | download from BOEM |
| `sale_261` | Sale 261 (Mar 2024) | download from BOEM |
| `sale_dec2025` | OBBBA Dec 2025 | download from BOEM |

In [ ]:
import os

# Example: download a specific sale's bid file directly from BOEM
os.makedirs('block-party/data/lease-sales/sale_257', exist_ok=True)
!wget -P block-party/data/lease-sales/sale_257 \
    "https://www.data.boem.gov/Leasing/Files/LeaseAucResults/Sale257Bids.zip"

# Unzip after downloading
!unzip -o block-party/data/lease-sales/sale_257/Sale257Bids.zip \
    -d block-party/data/lease-sales/sale_257/

In [ ]:
DATA = os.path.join(ROOT, "data")
LEASE_DIR = os.path.join(DATA, "lease-sales")
SHP_PATH = os.path.join(DATA, "shapefiles", "blocks.shp")

SALE_DIRS = {
    198: os.path.join(LEASE_DIR, "sale_198"),
    257: os.path.join(LEASE_DIR, "sale_257"),
    261: os.path.join(LEASE_DIR, "sale_261"),
    "dec2025": os.path.join(LEASE_DIR, "sale_dec2025"),
}

# Show which sale directories exist
for label, path in SALE_DIRS.items():
    status = "OK" if os.path.isdir(path) else "MISSING"
    print(f"  Sale {label:>8}  {status:>7}  {path}")

## 2. Load Sale 198 (sample data)

In [ ]:
s198 = bl.load_sale(SALE_DIRS[198])

print("Loaded tables:", list(s198.keys()))
for name, df in s198.items():
    print(f"  {name:12s}  {df.shape[0]:>5} rows  x  {df.shape[1]} cols")

In [ ]:
print("=== Tracts (PREBID) ===")
s198["tracts"].head()

In [ ]:
print("=== Bids ===")
s198["bids"].head()

In [ ]:
print("=== Companies ===")
s198["companies"].head()

In [ ]:
print("=== Maps (protraction area lookup) ===")
if "maps" in s198:
    display(s198["maps"].head())
else:
    print("(not present in this sale)")

### Sanity checks — Sale 198

In [ ]:
tracts = s198["tracts"]
bids = s198["bids"]
companies = s198["companies"]

# 1. Lease-number join coverage
tract_leases = set(tracts["Lease_Number"])
bid_leases = set(bids["Lease_Number"])
print(f"Unique tract leases : {len(tract_leases)}")
print(f"Unique bid leases   : {len(bid_leases)}")
print(f"Bid leases in tracts: {len(bid_leases & tract_leases)} / {len(bid_leases)}")

# 2. Company coverage
bid_cos = set(bids["Company_Number"])
known_cos = set(companies["Company_Number"])
print(f"\nUnique bidding companies: {len(bid_cos)}")
print(f"Companies in lookup    : {len(known_cos)}")
print(f"Bid companies matched  : {len(bid_cos & known_cos)} / {len(bid_cos)}")
missing = bid_cos - known_cos
if missing:
    print(f"  Unmatched company codes: {missing}")

# 3. Quick bid stats
print(f"\nTotal bid rows      : {len(bids)}")
print(f"Total bid amount ($): {bids['Bid_Amount'].sum():,.0f}")
print(f"Mean bid ($)        : {bids['Bid_Amount'].mean():,.0f}")
print(f"Tracts with >=2 bids: {(tracts['Num_Bids'] >= 2).sum()} / {len(tracts)}")

## 3. Load additional sales (uncomment once data is downloaded)

Download each sale's ZIP from https://data.boem.gov/Main/Leasing.aspx,
unzip into the matching `data/lease-sales/sale_<id>/` folder, then run these cells.

In [ ]:
# --- Sale 257 (Aug 2023) ---
# s257 = bl.load_sale(SALE_DIRS[257])
# print("Sale 257:", {k: v.shape for k, v in s257.items()})

In [ ]:
# --- Sale 261 (Mar 2024) ---
# s261 = bl.load_sale(SALE_DIRS[261])
# print("Sale 261:", {k: v.shape for k, v in s261.items()})

In [ ]:
# --- December 2025 OBBBA Sale (validation only) ---
# s_dec25 = bl.load_sale(SALE_DIRS["dec2025"])
# print("Dec 2025:", {k: v.shape for k, v in s_dec25.items()})

## 4. Load OCS block shapefile

In [ ]:
blocks = bl.load_blocks(SHP_PATH, to_utm=True)

print(f"Blocks loaded : {len(blocks):,}")
print(f"CRS           : {blocks.crs}")
print(f"Columns       : {list(blocks.columns)}")
blocks.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
blocks.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.2)
ax.set_title("GOM OCS Blocks (UTM 15N)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

### Spatial join check — Sale 198 tracts to blocks

In [ ]:
# Join tracts to blocks via (Protraction_ID, Block_Number)
merged = tracts.merge(
    blocks[["Protraction_ID", "Block_Number", "geometry"]],
    on=["Protraction_ID", "Block_Number"],
    how="left",
)
matched = merged["geometry"].notna().sum()
print(f"Tracts matched to blocks: {matched} / {len(tracts)}")

if matched < len(tracts):
    unmatched = merged[merged["geometry"].isna()][
        ["Lease_Number", "Protraction_ID", "Block_Number"]
    ]
    print("Sample unmatched tracts:")
    display(unmatched.head(10))

## 5. Supplemental datasets (placeholders)

Download lease, well, and relinquishment CSVs from BOEM, place under `data/`, and use these loaders.

In [ ]:
# Example — uncomment and set paths once files are downloaded:
#
# leases = bl.load_leases(os.path.join(DATA, "leases.csv"))
# wells  = bl.load_wells(os.path.join(DATA, "wells.csv"))
# relin  = bl.load_relinquishments(os.path.join(DATA, "relinquishments.csv"))
#
# print(f"Leases: {leases.shape}  Wells: {wells.shape}  Relinquishments: {relin.shape}")

---

**Next:** Use these loaded DataFrames in the investigation notebooks:
- `01_adjacency_signal.ipynb` (Q1)
- `02_relinquishment_signal.ipynb` (Q2)
- `03_well_activity_signal.ipynb` (Q3)
- `04_archetype_stability.ipynb` (Q4)

# MVP1

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from libpysal.weights import Rook

ROOT = '/content/block-party'
sys.path.insert(0, os.path.join(ROOT, 'mvp0'))
import boem_loader as bl

SALE_DATE = pd.Timestamp("2017-03-22")
print(f"Reference sale date: {SALE_DATE.date()}")

In [ ]:
sales = bl.load_master_sales(os.path.join(ROOT, "data/lease-sales/master_lease_sales.csv"))
lh    = bl.load_lease_history(os.path.join(ROOT, "data/lease-sales/cleaned_lease_history.csv"))
lo    = bl.load_lease_owners(os.path.join(ROOT, "data/lease-sales/lseowndelimit.txt"))
blocks = bl.load_blocks(os.path.join(ROOT, "data/shapefiles/blocks.shp"), to_utm=True)

print(f"Sale bids: {len(sales)} rows, {sales['Lease_Number'].nunique()} unique blocks")
print(f"Lease history: {len(lh)} rows")
print(f"Lease owners: {len(lo)} rows")
print(f"Blocks: {len(blocks)} rows")

In [ ]:
# Protraction areas in this sale
sale_prots = set(sales["Protraction_ID"].unique())
print(f"Sale protraction areas: {len(sale_prots)}")

# All blocks in those protraction areas
universe = blocks[blocks["Protraction_ID"].isin(sale_prots)].copy()
universe = universe.reset_index(drop=True)
# Give each block a unique integer index for the adjacency matrix
universe["block_idx"] = range(len(universe))
print(f"Blocks in sale universe: {len(universe)}")

In [ ]:
# Use the canonical helper — avoids the TERMED/TERMIN typo and code duplication
active_leases = bl.active_leases_at(lh, lo, SALE_DATE)
print(f"Leases active at sale time (matched to owner): {len(active_leases)}")
print(active_leases["Lease_Status"].value_counts().to_string())


In [ ]:
# active_leases_at() already joined lh → lo; only Lease_Number gets a suffix.
# All other lh columns (Area_Code, Block_Number, …) keep their original names.
active_blocks = active_leases.merge(
    universe[["AREA_CODE", "Block_Number", "block_idx"]],
    left_on=["Area_Code", "Block_Number"],
    right_on=["AREA_CODE", "Block_Number"],
    how="inner",
)
print(f"Active leases on blocks in sale universe: {len(active_blocks)}")
print(f"Unique companies with active leases in area: {active_blocks['Company_Number'].nunique()}")


In [ ]:
%%time
w = Rook.from_dataframe(universe, use_index=False)
print(f"Adjacency weights built: {w.n} blocks, mean {w.mean_neighbors:.1f} neighbors")

# Build a fast lookup: block_idx → set of neighbor block_idxs
adj = {i: set(w.neighbors[i]) for i in range(w.n)}

In [ ]:
# Companies that bid in Sale 247
bid_companies = sales["Company_Number"].unique()
print(f"Companies that bid: {len(bid_companies)}")

# For each company, the set of block_idxs they hold leases on
company_held = (
    active_blocks.groupby("Company_Number")["block_idx"]
    .apply(set)
    .to_dict()
)

# Bid lookup: (Company_Number, Protraction_ID, Block_Number) → True
bid_set = set(
    zip(sales["Company_Number"], sales["Protraction_ID"], sales["Block_Number"])
)

# Block info for lookup
blk_info = universe[["block_idx", "Protraction_ID", "Block_Number"]].to_dict("records")

# Build rows
rows = []
for co in bid_companies:
    held = company_held.get(co, set())
    # Blocks adjacent to any held block
    adj_blocks = set()
    for h in held:
        adj_blocks.update(adj.get(h, set()))
    adj_blocks -= held  # exclude blocks the company already holds

    for blk in blk_info:
        idx = blk["block_idx"]
        if idx in held:
            continue  # skip blocks company already leases
        is_adj = idx in adj_blocks
        did_bid = (co, blk["Protraction_ID"], blk["Block_Number"]) in bid_set
        rows.append((co, idx, is_adj, did_bid))

matrix = pd.DataFrame(rows, columns=["Company", "block_idx", "adjacent", "did_bid"])
print(f"Analysis matrix: {len(matrix):,} (company × block) pairs")
print(f"  adjacent=True: {matrix['adjacent'].sum():,}")
print(f"  did_bid=True:  {matrix['did_bid'].sum():,}")

In [ ]:
ct = pd.crosstab(matrix["adjacent"], matrix["did_bid"], margins=True)
ct.index = ct.index.map({False: "No adjacency", True: "Has adjacency", "All": "All"})
ct.columns = ct.columns.map({False: "No bid", True: "Bid", "All": "Total"})
print("=== Contingency table ===")
display(ct)

# Bid rates
adj_yes = matrix[matrix["adjacent"]]
adj_no  = matrix[~matrix["adjacent"]]

rate_adj = adj_yes["did_bid"].mean()
rate_non = adj_no["did_bid"].mean()
lift = rate_adj / rate_non if rate_non > 0 else float("inf")

print(f"\nBid rate (adjacent):     {rate_adj:.4%}  ({adj_yes['did_bid'].sum()} / {len(adj_yes)})")
print(f"Bid rate (non-adjacent): {rate_non:.4%}  ({adj_no['did_bid'].sum()} / {len(adj_no)})")
print(f"Lift: {lift:.1f}×")

In [ ]:
rng = np.random.default_rng(42)
N_PERM = 10_000

# Collapse to company level: has_adj (bool) + bid_count per company
company_summary = (
    matrix.groupby("Company")
    .agg(has_adj=("adjacent", "any"), bids=("did_bid", "sum"), n=("did_bid", "count"))
    .reset_index()
)

observed_lift = lift

perm_lifts = []
for _ in range(N_PERM):
    shuffled_adj = rng.permutation(company_summary["has_adj"].values)
    r_adj = company_summary.loc[shuffled_adj, "bids"].sum() / company_summary.loc[shuffled_adj, "n"].sum()
    r_non = company_summary.loc[~shuffled_adj, "bids"].sum() / company_summary.loc[~shuffled_adj, "n"].sum()
    perm_lifts.append(r_adj / r_non if r_non > 0 else np.nan)

perm_lifts = np.array(perm_lifts)
p_perm = np.nanmean(perm_lifts >= observed_lift)

print(f"Observed lift:              {observed_lift:.2f}×")
print(f"Permutation p-value:        {p_perm:.4f}  (N={N_PERM:,} permutations)")
print(f"Permutation lift (95th %%): {np.nanpercentile(perm_lifts, 95):.2f}×")

if p_perm < 0.05:
    print("\n→ Statistically significant (permutation p < 0.05)")
else:
    print("\n→ NOT statistically significant (permutation p >= 0.05)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: bid rates
ax = axes[0]
rates = pd.Series({"Adjacent": rate_adj * 100, "Non-adjacent": rate_non * 100})
rates.plot.bar(ax=ax, color=["#2196F3", "#9E9E9E"], edgecolor="black")
ax.set_ylabel("Bid rate (%)")
ax.set_title(f"Adjacency Lift: {lift:.1f}×  (p = {p_value:.2e})")
ax.tick_params(axis="x", rotation=0)
for i, v in enumerate(rates):
    ax.text(i, v + 0.1, f"{v:.2f}%", ha="center", fontsize=10)

# Map: blocks colored by adjacency status
ax = axes[1]
universe.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.2)

# Highlight blocks with active leases
held_idxs = set()
for s in company_held.values():
    held_idxs.update(s)
held_blocks = universe[universe["block_idx"].isin(held_idxs)]
held_blocks.plot(ax=ax, color="#FFC107", edgecolor="gray", linewidth=0.3, label="Active lease")

# Highlight bid blocks
bid_merged = sales.merge(
    universe[["Protraction_ID", "Block_Number", "geometry"]],
    on=["Protraction_ID", "Block_Number"], how="inner",
)
bid_gdf = gpd.GeoDataFrame(bid_merged, geometry="geometry")
bid_gdf.plot(ax=ax, color="#F44336", edgecolor="black", linewidth=0.5, label="Bid placed")

ax.set_title("Sale 247 — Active Leases & Bids")
ax.legend(loc="lower left", fontsize=8)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

## Notebook 02

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from libpysal.weights import Rook

ROOT = "/content/block-party"  # Colab path; update if running locally
sys.path.insert(0, os.path.join(ROOT, "mvp0"))
import boem_loader as bl

SALE_DATE = pd.Timestamp("2017-03-22")
LOOKBACK_MONTHS = 36
LOOKBACK_START = SALE_DATE - pd.DateOffset(months=LOOKBACK_MONTHS)
print(f"Sale date: {SALE_DATE.date()}")
print(f"Relinquishment window: {LOOKBACK_START.date()} → {SALE_DATE.date()}")

In [ ]:
sales  = bl.load_master_sales(os.path.join(ROOT, "data/lease-sales/master_lease_sales.csv"))
lh     = bl.load_lease_history(os.path.join(ROOT, "data/lease-sales/cleaned_lease_history.csv"))
blocks = bl.load_blocks(os.path.join(ROOT, "data/shapefiles/blocks.shp"), to_utm=True)
wells  = bl.load_boreholes(os.path.join(ROOT, "data/wells/mv_boreholes_all.txt"), region="G", to_utm=True)

print(f"Sale bids: {len(sales)}")
print(f"Lease history: {len(lh)}")
print(f"Blocks: {len(blocks)}")
print(f"Wells (for water depth): {len(wells)}")


In [ ]:
sale_prots = set(sales["Protraction_ID"].unique())
universe = blocks[blocks["Protraction_ID"].isin(sale_prots)].copy().reset_index(drop=True)
universe["block_idx"] = range(len(universe))
print(f"Universe: {len(universe)} blocks in {len(sale_prots)} protraction areas")

# Rook adjacency
w = Rook.from_dataframe(universe, use_index=False)
adj = {i: set(w.neighbors[i]) for i in range(w.n)}
print(f"Adjacency: mean {w.mean_neighbors:.1f} neighbors")

In [ ]:
relin = lh[
    (lh["Lease_Status"] == "RELINQ") &
    (lh["Status_Date"] >= LOOKBACK_START) &
    (lh["Status_Date"] <= SALE_DATE)
].copy()
print(f"Relinquishments in 36-month window: {len(relin)}")

# Join to blocks in universe
relin_blocks = relin.merge(
    universe[["AREA_CODE", "Block_Number", "block_idx"]],
    left_on=["Area_Code", "Block_Number"],
    right_on=["AREA_CODE", "Block_Number"],
    how="inner",
)
print(f"Relinquished blocks in sale universe: {relin_blocks['block_idx'].nunique()}")

relin_idxs = set(relin_blocks["block_idx"])

In [ ]:
# Blocks adjacent to any relinquished block
relin_adj_idxs = set()
for ri in relin_idxs:
    relin_adj_idxs.update(adj.get(ri, set()))
relin_adj_idxs -= relin_idxs  # exclude the relinquished blocks themselves

universe["relin_adjacent"] = universe["block_idx"].isin(relin_adj_idxs)
print(f"Blocks adjacent to relinquishment: {universe['relin_adjacent'].sum()}")
print(f"Blocks NOT adjacent: {(~universe['relin_adjacent']).sum()}")

In [ ]:
bid_pairs = set(zip(sales["Protraction_ID"], sales["Block_Number"]))
universe["did_bid"] = [
    (r["Protraction_ID"], r["Block_Number"]) in bid_pairs
    for _, r in universe.iterrows()
]
print(f"Blocks with bids: {universe['did_bid'].sum()}")

In [ ]:
ct = pd.crosstab(universe["relin_adjacent"], universe["did_bid"], margins=True)
ct.index = ct.index.map({False: "No relin. adjacent", True: "Relin. adjacent", "All": "All"})
ct.columns = ct.columns.map({False: "No bid", True: "Bid", "All": "Total"})
display(ct)

rate_relin = universe.loc[universe["relin_adjacent"], "did_bid"].mean()
rate_no_relin = universe.loc[~universe["relin_adjacent"], "did_bid"].mean()
ratio = rate_relin / rate_no_relin if rate_no_relin > 0 else float("inf")

print(f"\nBid rate (relin-adjacent):     {rate_relin:.4%}")
print(f"Bid rate (no relin-adjacent):  {rate_no_relin:.4%}")
print(f"Ratio: {ratio:.2f}  (< 0.50 = cold zone confirmed)")

# Chi-square
ct_vals = pd.crosstab(universe["relin_adjacent"], universe["did_bid"])
chi2, p_val, dof, _ = chi2_contingency(ct_vals)
print(f"Chi-square: {chi2:.2f}, p = {p_val:.2e}")

In [ ]:
# Median water depth per (Area_Code, Block_Number) from borehole data
well_depth = (
    wells.groupby(["Area_Code", "Block_Number"])["Water_Depth"]
    .median()
    .reset_index()
    .rename(columns={"Water_Depth": "Water_Depth_ft"})
)
well_depth["Water_Depth_m"] = well_depth["Water_Depth_ft"] * 0.3048

universe = universe.merge(
    well_depth[["Area_Code", "Block_Number", "Water_Depth_m"]],
    left_on=["AREA_CODE", "Block_Number"],
    right_on=["Area_Code", "Block_Number"],
    how="left",
).drop(columns=["Area_Code"])

# Depth buckets
universe["depth_bucket"] = pd.cut(
    universe["Water_Depth_m"],
    bins=[0, 200, 1500, 99999],
    labels=["Shelf (<200m)", "Deep (200–1500m)", "Ultra-deep (>1500m)"],
)

has_depth = universe["depth_bucket"].notna()
print(f"Blocks with depth data: {has_depth.sum()} / {len(universe)}")
print(universe["depth_bucket"].value_counts().to_string())

In [ ]:
# Bid rates by depth bucket × relinquishment adjacency
depth_analysis = (
    universe[has_depth]
    .groupby(["depth_bucket", "relin_adjacent"])["did_bid"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "bids", "count": "blocks", "mean": "bid_rate"})
)
print("Bid rates by depth and relinquishment adjacency:")
display(depth_analysis)

In [ ]:
# Split relinquishments into recent (0-12mo) and older (13-36mo)
cutoff_12mo = SALE_DATE - pd.DateOffset(months=12)

relin_recent = relin_blocks[relin_blocks["Status_Date"] >= cutoff_12mo]
relin_older  = relin_blocks[relin_blocks["Status_Date"] < cutoff_12mo]

recent_adj = set()
for ri in set(relin_recent["block_idx"]):
    recent_adj.update(adj.get(ri, set()))

older_adj = set()
for ri in set(relin_older["block_idx"]):
    older_adj.update(adj.get(ri, set()))

universe["relin_recent"] = universe["block_idx"].isin(recent_adj - relin_idxs)
universe["relin_older"]  = universe["block_idx"].isin(older_adj - relin_idxs)

decay = pd.DataFrame({
    "Window": ["0–12 months", "13–36 months", "No relinquishment"],
    "Blocks": [
        universe["relin_recent"].sum(),
        universe["relin_older"].sum(),
        (~universe["relin_adjacent"]).sum(),
    ],
    "Bid rate": [
        universe.loc[universe["relin_recent"], "did_bid"].mean(),
        universe.loc[universe["relin_older"], "did_bid"].mean(),
        rate_no_relin,
    ],
})
print("Relinquishment decay:")
display(decay)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart: overall bid rates
ax = axes[0]
rates = pd.Series({"Relin. adjacent": rate_relin * 100, "Not adjacent": rate_no_relin * 100})
rates.plot.bar(ax=ax, color=["#F44336", "#4CAF50"], edgecolor="black")
ax.set_ylabel("Bid rate (%)")
ax.set_title(f"Relinquishment Effect (ratio = {ratio:.2f})")
ax.tick_params(axis="x", rotation=0)

# Decay curve
ax = axes[1]
ax.bar(decay["Window"], decay["Bid rate"] * 100, color=["#F44336", "#FF9800", "#4CAF50"], edgecolor="black")
ax.set_ylabel("Bid rate (%)")
ax.set_title("Relinquishment Signal Decay")
ax.tick_params(axis="x", rotation=15)

# Map
ax = axes[2]
universe.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.2)
relin_gdf = universe[universe["block_idx"].isin(relin_idxs)]
relin_gdf.plot(ax=ax, color="#F44336", edgecolor="black", linewidth=0.3, label="Relinquished")
adj_gdf = universe[universe["relin_adjacent"]]
adj_gdf.plot(ax=ax, color="#FFCDD2", edgecolor="gray", linewidth=0.2, label="Adjacent to relin.")
bid_blocks = universe[universe["did_bid"]]
bid_blocks.plot(ax=ax, color="#2196F3", edgecolor="black", linewidth=0.5, label="Bid placed")
ax.set_title("Relinquishments & Bids")
ax.legend(loc="lower left", fontsize=7)

plt.tight_layout()
plt.show()

## 03 Notebooks

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

ROOT = "/content/block-party"  # Colab path; update if running locally
sys.path.insert(0, os.path.join(ROOT, "mvp0"))
import boem_loader as bl

SALE_DATE = pd.Timestamp("2017-03-22")
RADII_KM = [10, 25]
WINDOWS_MO = [6, 18]
print(f"Sale date: {SALE_DATE.date()}")
print(f"Radii: {RADII_KM} km")
print(f"Lookback windows: {WINDOWS_MO} months")

In [ ]:
sale_prots = set(sales["Protraction_ID"].unique())
universe = blocks[blocks["Protraction_ID"].isin(sale_prots)].copy().reset_index(drop=True)
print(f"Universe: {len(universe)} blocks")

# Mark which blocks received bids
bid_pairs = set(zip(sales["Protraction_ID"], sales["Block_Number"]))
universe["did_bid"] = [
    (r["Protraction_ID"], r["Block_Number"]) in bid_pairs
    for _, r in universe.iterrows()
]
print(f"Blocks with bids: {universe['did_bid'].sum()}")

# Block centroids (UTM meters) for distance calculations
universe["centroid"] = universe.geometry.centroid

In [ ]:
well_sets = {}
for mo in WINDOWS_MO:
    start = SALE_DATE - pd.DateOffset(months=mo)
    mask = (wells["Spud_Date"] >= start) & (wells["Spud_Date"] <= SALE_DATE)
    well_sets[mo] = wells[mask].copy()
    print(f"Wells spudded in prior {mo} months: {len(well_sets[mo])}")

In [ ]:
%%time
# Create centroid GeoDataFrame for spatial joins
centroids = gpd.GeoDataFrame(
    universe[["Protraction_ID", "Block_Number", "did_bid"]],
    geometry=universe["centroid"],
    crs=universe.crs,
)

results = {}
for mo in WINDOWS_MO:
    ws = well_sets[mo]
    for radius_km in RADII_KM:
        radius_m = radius_km * 1000
        label = f"{radius_km}km_{mo}mo"

        # Buffer wells
        buffered = ws.copy()
        buffered["geometry"] = buffered.geometry.buffer(radius_m)

        # Spatial join: which block centroids fall within each well buffer?
        joined = gpd.sjoin(centroids, buffered[["geometry"]], how="left", predicate="within")

        # Count wells per block
        well_count = (
            joined.groupby(joined.index)["index_right"]
            .count()
            .reindex(range(len(universe)), fill_value=0)
        )
        universe[f"wells_{label}"] = well_count.values
        results[label] = universe[f"wells_{label}"]

        n_with = (well_count > 0).sum()
        print(f"{label}: {n_with} blocks have >= 1 well nearby")

In [ ]:
combo_results = []

for label in results:
    col = f"wells_{label}"
    universe["well_bin"] = pd.cut(
        universe[col], bins=[-1, 0, 1, 999], labels=["0 wells", "1 well", "2+ wells"]
    )

    rates = universe.groupby("well_bin", observed=True)["did_bid"].agg(["sum", "count", "mean"])
    rates.columns = ["bids", "blocks", "bid_rate"]

    # Spearman correlation between well count and bid
    mask = universe[col].notna()
    rho, p = spearmanr(universe.loc[mask, col], universe.loc[mask, "did_bid"])

    combo_results.append({
        "combination": label,
        "rate_0": rates.loc["0 wells", "bid_rate"] if "0 wells" in rates.index else 0,
        "rate_1": rates.loc["1 well", "bid_rate"] if "1 well" in rates.index else 0,
        "rate_2plus": rates.loc["2+ wells", "bid_rate"] if "2+ wells" in rates.index else 0,
        "spearman_rho": rho,
        "p_value": p,
    })

    print(f"\n=== {label} ===")
    display(rates)
    print(f"Spearman rho = {rho:.4f}, p = {p:.2e}")

In [ ]:
summary = pd.DataFrame(combo_results)
display(summary)

# Find strongest combination
best = summary.loc[summary["spearman_rho"].abs().idxmax()]
print(f"\nStrongest combination: {best['combination']}")
print(f"  Spearman rho = {best['spearman_rho']:.4f}, p = {best['p_value']:.2e}")

In [ ]:
# Heatmap of bid rates by (radius, window, well_bin)
fig, axes = plt.subplots(1, len(RADII_KM), figsize=(14, 5), sharey=True)

for i, radius_km in enumerate(RADII_KM):
    ax = axes[i]
    for mo in WINDOWS_MO:
        label = f"{radius_km}km_{mo}mo"
        col = f"wells_{label}"
        universe["well_bin"] = pd.cut(
            universe[col], bins=[-1, 0, 1, 999], labels=["0", "1", "2+"]
        )
        rates = universe.groupby("well_bin", observed=True)["did_bid"].mean() * 100
        ax.plot(rates.index, rates.values, "o-", label=f"{mo}-month window", markersize=8)

    ax.set_title(f"Radius: {radius_km} km")
    ax.set_xlabel("Wells in radius")
    ax.set_ylabel("Bid rate (%)" if i == 0 else "")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Well Activity Signal: Bid Rate by Nearby Well Count", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Use the best radius, split the 18-month window into recent vs. older
best_radius = int(best["combination"].split("km")[0])
radius_m = best_radius * 1000

cutoff_6mo = SALE_DATE - pd.DateOffset(months=6)
cutoff_18mo = SALE_DATE - pd.DateOffset(months=18)

recent_wells = wells[(wells["Spud_Date"] >= cutoff_6mo) & (wells["Spud_Date"] <= SALE_DATE)]
older_wells  = wells[(wells["Spud_Date"] >= cutoff_18mo) & (wells["Spud_Date"] < cutoff_6mo)]

for lag_label, ws in [("0–6 months", recent_wells), ("7–18 months", older_wells)]:
    buffered = ws.copy()
    buffered["geometry"] = buffered.geometry.buffer(radius_m)
    joined = gpd.sjoin(centroids, buffered[["geometry"]], how="left", predicate="within")
    wc = joined.groupby(joined.index)["index_right"].count().reindex(range(len(universe)), fill_value=0)
    has_well = wc > 0
    rate_with = universe.loc[has_well.values, "did_bid"].mean()
    rate_without = universe.loc[~has_well.values, "did_bid"].mean()
    print(f"{lag_label} ({best_radius}km): with_well={rate_with:.4%} vs without={rate_without:.4%}  "
          f"(lift={rate_with/rate_without:.1f}x)" if rate_without > 0 else "")

In [ ]:
best_col = f"wells_{best['combination']}"

fig, ax = plt.subplots(figsize=(12, 8))
universe.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.2)

# Shade blocks by well count
has_wells = universe[universe[best_col] > 0]
has_wells.plot(ax=ax, column=best_col, cmap="YlOrRd", edgecolor="gray",
               linewidth=0.3, legend=True, legend_kwds={"label": "Wells in radius"})

# Overlay bid blocks
bid_blocks = universe[universe["did_bid"]]
bid_blocks.plot(ax=ax, facecolor="none", edgecolor="blue", linewidth=1.5, label="Bid placed")

# Plot well locations
best_mo = int(best["combination"].split("_")[1].replace("mo", ""))
ws = well_sets[best_mo]
ws.plot(ax=ax, color="black", markersize=3, alpha=0.5, label="Wells")

ax.set_title(f"Well Activity Signal — {best['combination']} (Sale 247)")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## 05 Notebook

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from libpysal.weights import Rook
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
from sklearn.preprocessing import StandardScaler

ROOT = "/content/block-party"  # Colab path; update if running locally
sys.path.insert(0, os.path.join(ROOT, "mvp0"))
import boem_loader as bl

SALE_DATE = pd.Timestamp("2017-03-22")
WELL_RADIUS_KM = 25
WELL_WINDOW_MO = 6
print(f"Sale date: {SALE_DATE.date()}")
print(f"Well window: {WELL_WINDOW_MO} months, radius: {WELL_RADIUS_KM} km")

In [ ]:
sales  = bl.load_master_sales(os.path.join(ROOT, "data/lease-sales/master_lease_sales.csv"))
lh     = bl.load_lease_history(os.path.join(ROOT, "data/lease-sales/cleaned_lease_history.csv"))
lo     = bl.load_lease_owners(os.path.join(ROOT, "data/lease-sales/lseowndelimit.txt"))
blocks = bl.load_blocks(os.path.join(ROOT, "data/shapefiles/blocks.shp"), to_utm=True)
wells  = bl.load_boreholes(os.path.join(ROOT, "data/wells/mv_boreholes_all.txt"), region="G", to_utm=True)

print(f"Sale bids: {len(sales)} | Lease history: {len(lh)} | Owners: {len(lo)}")
print(f"Blocks: {len(blocks)} | Wells: {len(wells)}")


In [ ]:
sale_prots = set(sales["Protraction_ID"].unique())
universe = blocks[blocks["Protraction_ID"].isin(sale_prots)].copy().reset_index(drop=True)
universe["block_idx"] = range(len(universe))

bid_pairs = set(zip(sales["Protraction_ID"], sales["Block_Number"]))
universe["did_bid"] = [
    (r["Protraction_ID"], r["Block_Number"]) in bid_pairs
    for _, r in universe.iterrows()
]
print(f"Universe: {len(universe)} blocks, {universe['did_bid'].sum()} with bids")

In [ ]:
# Use the canonical helper — avoids the TERMED/TERMIN typo and code duplication
active_co = bl.active_leases_at(lh, lo, SALE_DATE)

# Rename clashing columns produced by the inner merge
if "Area_Code_lh" in active_co.columns:
    active_co = active_co.rename(columns={"Area_Code_lh": "Area_Code"})

active_blocks = active_co.merge(
    universe[["AREA_CODE", "Block_Number", "block_idx"]],
    left_on=["Area_Code", "Block_Number"],
    right_on=["AREA_CODE", "Block_Number"],
    how="inner",
)
print(f"Active leases in universe: {len(active_blocks)}")
print(f"Unique companies: {active_blocks['Company_Number'].nunique()}")


In [ ]:
# Rook adjacency
w = Rook.from_dataframe(universe, use_index=False)
adj = {i: set(w.neighbors[i]) for i in range(w.n)}

# For each block, count distinct companies on adjacent blocks
# Build: block_idx → set of Company_Numbers holding leases
block_companies = (
    active_blocks.groupby("block_idx")["Company_Number"]
    .apply(set)
    .to_dict()
)

adj_company_counts = []
for idx in range(len(universe)):
    neighbor_cos = set()
    for ni in adj.get(idx, set()):
        neighbor_cos.update(block_companies.get(ni, set()))
    adj_company_counts.append(len(neighbor_cos))

universe["n_adj_companies"] = adj_company_counts

print(f"Blocks with >= 1 adjacent company: {(universe['n_adj_companies'] > 0).sum()}")
print(universe["n_adj_companies"].describe().to_string())

In [ ]:
well_start = SALE_DATE - pd.DateOffset(months=WELL_WINDOW_MO)
recent_wells = wells[
    (wells["Spud_Date"] >= well_start) & (wells["Spud_Date"] <= SALE_DATE)
].copy()
print(f"Wells in {WELL_WINDOW_MO}-month window: {len(recent_wells)}")

# Buffer wells and spatial join to block centroids
centroids = gpd.GeoDataFrame(
    universe[["block_idx"]],
    geometry=universe.geometry.centroid,
    crs=universe.crs,
)

buffered = recent_wells.copy()
buffered["geometry"] = buffered.geometry.buffer(WELL_RADIUS_KM * 1000)

joined = gpd.sjoin(centroids, buffered[["geometry"]], how="left", predicate="within")
well_count = (
    joined.groupby(joined.index)["index_right"]
    .count()
    .reindex(range(len(universe)), fill_value=0)
)
universe["n_wells_nearby"] = well_count.values

print(f"Blocks with >= 1 well nearby: {(universe['n_wells_nearby'] > 0).sum()}")
print(universe["n_wells_nearby"].describe().to_string())

In [ ]:
features = ["n_adj_companies", "n_wells_nearby"]
X = universe[features].values
y = universe["did_bid"].astype(int).values

print(f"Feature matrix: {X.shape[0]} blocks × {X.shape[1]} features")
print(f"Positive class (bid): {y.sum()} ({y.mean():.2%})")
print(f"Negative class (no bid): {(1 - y).sum()}")

# Correlation between features
corr = np.corrcoef(X[:, 0], X[:, 1])[0, 1]
print(f"\nPearson correlation between features: {corr:.3f}")

# Per-feature bid rate lift
for feat in features:
    has = universe[universe[feat] > 0]["did_bid"].mean()
    has_not = universe[universe[feat] == 0]["did_bid"].mean()
    lift = has / has_not if has_not > 0 else float("inf")
    print(f"{feat}: bid rate {has:.2%} (present) vs {has_not:.2%} (absent) → {lift:.1f}x lift")

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

rng = np.random.default_rng(42)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# class_weight="balanced" compensates for the ~50:1 class imbalance
model = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
model.fit(X_scaled, y)

# In-sample score (optimistic upper bound)
universe["score"] = model.predict_proba(X_scaled)[:, 1]

# Cross-validated metrics (more honest)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = cross_val_score(model, X_scaled, y, cv=cv, scoring="roc_auc")
cv_aps  = cross_val_score(model, X_scaled, y, cv=cv, scoring="average_precision")

print("Logistic regression coefficients (standardized):")
for feat, coef in zip(features, model.coef_[0]):
    print(f"  {feat:20s}: {coef:+.4f}")
print(f"  {'intercept':20s}: {model.intercept_[0]:+.4f}")
print(f"\nClass balance: {y.sum()} bids / {len(y)} blocks = {y.mean():.2%}")
print(f"\n=== In-sample (optimistic) ===")
print(f"  ROC-AUC:       {roc_auc_score(y, universe['score']):.4f}")
print(f"  Avg Precision: {average_precision_score(y, universe['score']):.4f}  (baseline: {y.mean():.4f})")
print(f"\n=== 5-fold CV (more honest) ===")
print(f"  ROC-AUC:       {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}")
print(f"  Avg Precision: {cv_aps.mean():.4f} ± {cv_aps.std():.4f}")


In [ ]:
auc = roc_auc_score(y, universe["score"])
ap = average_precision_score(y, universe["score"])
print(f"ROC-AUC:          {auc:.4f}")
print(f"Avg Precision:    {ap:.4f}")

# Individual feature AUCs for comparison
for i, feat in enumerate(features):
    feat_auc = roc_auc_score(y, X[:, i])
    print(f"AUC ({feat} alone): {feat_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
ax = axes[0]
fpr, tpr, _ = roc_curve(y, universe["score"])
ax.plot(fpr, tpr, "b-", linewidth=2, label=f"Combined (AUC={auc:.3f})")
for i, feat in enumerate(features):
    feat_fpr, feat_tpr, _ = roc_curve(y, X[:, i])
    feat_auc = roc_auc_score(y, X[:, i])
    ax.plot(feat_fpr, feat_tpr, "--", linewidth=1, label=f"{feat} (AUC={feat_auc:.3f})")
ax.plot([0, 1], [0, 1], "k:", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Precision-Recall curve
ax = axes[1]
prec, rec, _ = precision_recall_curve(y, universe["score"])
ax.plot(rec, prec, "b-", linewidth=2, label=f"Combined (AP={ap:.3f})")
for i, feat in enumerate(features):
    feat_prec, feat_rec, _ = precision_recall_curve(y, X[:, i])
    feat_ap = average_precision_score(y, X[:, i])
    ax.plot(feat_rec, feat_prec, "--", linewidth=1, label=f"{feat} (AP={feat_ap:.3f})")
baseline = y.mean()
ax.axhline(baseline, color="k", linestyle=":", alpha=0.4, label=f"Baseline ({baseline:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle("Combined Score: Q1 Adjacency + Q3 Well Activity", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
ranked = universe.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = range(1, len(ranked) + 1)
ranked["cum_bids"] = ranked["did_bid"].cumsum()

total_bids = y.sum()

top_ns = [50, 100, 200, 300, 500, 1000]
print(f"{'Top-N':>8}  {'Precision':>10}  {'Recall':>8}  {'Bids found':>11}")
print("-" * 45)
for n in top_ns:
    if n > len(ranked):
        continue
    top = ranked.head(n)
    prec_n = top["did_bid"].mean()
    recall_n = top["did_bid"].sum() / total_bids
    print(f"{n:>8}  {prec_n:>10.1%}  {recall_n:>8.1%}  {int(top['did_bid'].sum()):>6} / {total_bids}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ns = np.arange(1, len(ranked) + 1)
cum_recall = ranked["cum_bids"].values / total_bids
cum_precision = ranked["cum_bids"].values / ns

ax.plot(ns, cum_recall * 100, "b-", linewidth=2, label="Recall (% bids captured)")
ax.plot(ns, cum_precision * 100, "r-", linewidth=2, label="Precision (% flagged that are bids)")
ax.axhline(baseline * 100, color="gray", linestyle=":", label=f"Random precision ({baseline:.1%})")

ax.set_xlabel("Blocks flagged (top-N by score)")
ax.set_ylabel("%")
ax.set_title("Precision & Recall vs. number of blocks flagged")
ax.set_xlim(0, 2000)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Score distribution by class
ax = axes[0]
ax.hist(universe.loc[universe["did_bid"], "score"], bins=40, alpha=0.7,
        color="#2196F3", label="Bid", density=True)
ax.hist(universe.loc[~universe["did_bid"], "score"], bins=40, alpha=0.5,
        color="#9E9E9E", label="No bid", density=True)
ax.set_xlabel("Predicted score")
ax.set_ylabel("Density")
ax.set_title("Score distribution")
ax.legend()

# Scatter: feature 1 vs feature 2, colored by bid
ax = axes[1]
no_bid = universe[~universe["did_bid"]]
bid = universe[universe["did_bid"]]
ax.scatter(no_bid["n_adj_companies"], no_bid["n_wells_nearby"],
           c="#BDBDBD", s=5, alpha=0.3, label="No bid")
ax.scatter(bid["n_adj_companies"], bid["n_wells_nearby"],
           c="#F44336", s=30, edgecolors="black", linewidth=0.5, label="Bid", zorder=5)
ax.set_xlabel("Adjacent companies")
ax.set_ylabel("Wells nearby (25km/6mo)")
ax.set_title("Feature space")
ax.legend(fontsize=8)

# Coefficient bar chart
ax = axes[2]
coefs = pd.Series(model.coef_[0], index=features)
coefs.plot.barh(ax=ax, color=["#2196F3", "#4CAF50"], edgecolor="black")
ax.set_xlabel("Logistic regression coefficient (standardized)")
ax.set_title("Feature importance")

plt.suptitle("Combined Score Analysis", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Flag top-200 blocks as predictions
TOP_N = 200
top_idxs = set(ranked.head(TOP_N).index)
universe["top_predicted"] = universe.index.isin(top_idxs)

fig, ax = plt.subplots(figsize=(14, 9))
universe.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.1)

# Predicted blocks
predicted = universe[universe["top_predicted"]]
predicted.plot(ax=ax, color="#BBDEFB", edgecolor="#1565C0", linewidth=0.4, label=f"Top {TOP_N} predicted")

# True positive = predicted + bid
tp = universe[universe["top_predicted"] & universe["did_bid"]]
tp.plot(ax=ax, color="#2196F3", edgecolor="black", linewidth=0.8, label="True positive (predicted + bid)")

# False negative = bid but not predicted
fn = universe[~universe["top_predicted"] & universe["did_bid"]]
fn.plot(ax=ax, color="#F44336", edgecolor="black", linewidth=0.8, label="Missed bid (not in top-N)")

prec_top = tp.shape[0] / TOP_N if TOP_N > 0 else 0
recall_top = tp.shape[0] / total_bids if total_bids > 0 else 0

ax.set_title(f"Top-{TOP_N} Prediction Map — Precision {prec_top:.0%}, Recall {recall_top:.0%}")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()